## 1. Imports and repository paths

In [ ]:
from src import utils
from pathlib import Path
import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
import optuna
from IPython.display import display

# Support running the notebook from either the repository root or notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (CWD, CWD.parent) if (p / "src" / "main.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open JupyterLab in the project root or its notebooks/ directory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.main import RecommendationModel, SplitData

logging.getLogger("src.main").setLevel(logging.WARNING)  # Avoid per-epoch messages during tuning.
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Project: {PROJECT_ROOT}")

/Users/ayanful/miniconda3/envs/movie_rec/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /Users/ayanful/Library/CloudStorage/GoogleDrive-emmanuelayanful@u.boisestate.edu/My Drive/Academic Work/Research/CompExamFolder/Code Repo/Movie-Recommendation-System


## 2. Configuration

In [2]:
# Change only these experiment settings to run on another dataset.
DATASET_NAME = "ml-32m"         # Change to "ml-latest-small" for the small experiment.
N_FACTORS = 16                  # Fixed K, matching main.py's --n-factors.
N_EPOCHS = 20                   # Match the --n-epochs used for final training.
N_TRIALS = 20                   # Use 2 for a quick smoke test.
SEED = 42
TEST_SIZE = 0.20                # Used only if the original split needs to be built.
VALIDATION_FRACTION = 0.10      # Fraction of original TRAINING ratings for validation.

DATA_DIR = PROJECT_ROOT / "data"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments"

# Used solely for consistent training and path management; tuning does not
# call train_model(), which would overwrite final trained-model artifacts.
runner = RecommendationModel(
    n_factors=N_FACTORS,
    n_epochs=N_EPOCHS,
    seed=SEED,
    test_size=TEST_SIZE,
    dataset_name=DATASET_NAME,
    dataset_dir=DATA_DIR,
    experiment_dir=EXPERIMENT_DIR,
)
runner.model_path.mkdir(parents=True, exist_ok=True)
image_path = runner.experiment_path / "images"
image_path.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Tuning epochs: {N_EPOCHS}; trials/model: {N_TRIALS}; factors: {N_FACTORS}")
print(f"Best parameter files: {runner.model_path}")

Dataset: ml-32m
Tuning epochs: 20; trials/model: 20; factors: 16
Best parameter files: /Users/ayanful/Library/CloudStorage/GoogleDrive-emmanuelayanful@u.boisestate.edu/My Drive/Academic Work/Research/CompExamFolder/Code Repo/Movie-Recommendation-System/experiments/ml-32m/model_files


## 3. Prepare training artifacts and construct a separate validation set

In [3]:
# Reuse existing downloaded and preprocessed artifacts when available.
runner.preprocess_data(force=False)

preprocess_path = DATA_DIR / DATASET_NAME / "preprocess"
train_groups = utils.load_numpy_array(
    str(runner.split_path), "users_train", log=False
)
train_user_ids = utils.load_numpy_array(
    str(runner.split_path), "users_indices_train", log=False
)
n_users = len(utils.load_data_from_pickle(
    str(preprocess_path), "map_user_to_id", log=False
))
n_movies = len(utils.load_data_from_pickle(
    str(preprocess_path), "map_movie_to_id", log=False
))

print(f"Original training groups: {len(train_user_ids):,} users")
print(f"Full mapping: {n_users:,} users, {n_movies:,} movies")

Original training groups: 200,948 users
Full mapping: 200,948 users, 84,432 movies


### Validation splitting helper

The helper below creates a fresh train/validation split **only from `users_train.npy`**. It keeps at least one fitting observation for each user and movie represented in the original training data, preventing avoidable cold-start entities in validation. The resulting grouped arrays have exactly the format required by `models.py`.

Internally, `SplitData.users_test` contains *validation* observations while tuning; this is simply to reuse `main.py`'s existing training/evaluation methods. **The original held-out test split is never used for Optuna.**

In [4]:
def build_validation_split(
    original_groups: np.ndarray,
    original_user_ids: np.ndarray,
    n_users: int,
    n_movies: int,
    fraction: float = 0.10,
    seed: int = 42,
) -> SplitData:
    """
    Create a warm-start tuning train/validation split from original TRAIN ratings.

    The existing grouped (movie_indices, ratings) format is retained; only the
    *training* movies are regrouped by movie ID for the ALS movie-side sweep.
    """
    if not 0 < fraction < 1:
        raise ValueError("Validation fraction must be between 0 and 1.")
    if original_groups.ndim != 2 or original_groups.shape[1] != 2:
        raise ValueError(
            "Expected users_train.npy with shape (n_active_users, 2). "
            "Regenerate the split with main.py --force-preprocess if necessary."
        )
    original_user_ids = np.asarray(original_user_ids, dtype=np.int64).reshape(-1)
    if len(original_groups) != len(original_user_ids):
        raise ValueError("Training group rows and user ID array do not match.")

    counts = np.fromiter(
        (len(row[0]) for row in original_groups), dtype=np.int64,
        count=len(original_groups),
    )
    boundaries = np.empty(len(counts) + 1, dtype=np.int64)
    boundaries[0] = 0
    np.cumsum(counts, out=boundaries[1:])
    if boundaries[-1] < 2:
        raise ValueError("Not enough training ratings to create validation data.")

    movie_ids = np.concatenate([
        np.asarray(row[0], dtype=np.int64) for row in original_groups
    ])
    ratings = np.concatenate([
        np.asarray(row[1], dtype=np.float64) for row in original_groups
    ])
    if len(movie_ids) != len(ratings):
        raise ValueError("Movie indices and ratings are not aligned.")

    rng = np.random.default_rng(seed)
    validation_mask = rng.random(len(movie_ids)) < fraction

    # Every user with original training ratings retains >= 1 fitting rating.
    for start, stop in zip(boundaries[:-1], boundaries[1:]):
        if validation_mask[start:stop].all():
            validation_mask[start] = False

    # Every movie observed in the original training set retains >= 1 fitting rating.
    fitted_movie_count = np.bincount(movie_ids[~validation_mask], minlength=n_movies)
    need_movie = validation_mask & (fitted_movie_count[movie_ids] == 0)
    if need_movie.any():
        _, first_positions = np.unique(movie_ids[need_movie], return_index=True)
        validation_positions = np.flatnonzero(need_movie)
        validation_mask[validation_positions[first_positions]] = False

    if not validation_mask.any():
        raise ValueError(
            "The validation set is empty; increase the number of ratings or "
            "validation fraction."
        )

    def grouped_by_user(keep: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        active_ids = []
        items = []
        values = []
        for idx, (start, stop) in enumerate(zip(boundaries[:-1], boundaries[1:])):
            selected = keep[start:stop]
            if selected.any():
                active_ids.append(original_user_ids[idx])
                items.append(movie_ids[start:stop][selected].copy())
                values.append(ratings[start:stop][selected].copy())
        result = np.empty((len(active_ids), 2), dtype=object)
        for idx in range(len(active_ids)):
            result[idx, 0] = items[idx]
            result[idx, 1] = values[idx]
        return np.asarray(active_ids, dtype=np.int64), result

    fit_user_ids, fit_users = grouped_by_user(~validation_mask)
    validation_user_ids, validation_users = grouped_by_user(validation_mask)

    # Regroup the fitting interactions by movie for the movie-side ALS updates.
    all_user_ids = np.repeat(original_user_ids, counts)
    fit_movies_flat = movie_ids[~validation_mask]
    fit_users_flat = all_user_ids[~validation_mask]
    fit_ratings_flat = ratings[~validation_mask]
    order = np.argsort(fit_movies_flat, kind="stable")
    sorted_movies = fit_movies_flat[order]
    sorted_users = fit_users_flat[order]
    sorted_ratings = fit_ratings_flat[order]
    starts = np.r_[0, np.flatnonzero(np.diff(sorted_movies)) + 1]
    ends = np.r_[starts[1:], len(sorted_movies)]
    fit_movie_ids = sorted_movies[starts].copy()
    fit_movies = np.empty((len(starts), 2), dtype=object)
    for row, (start, stop) in enumerate(zip(starts, ends)):
        fit_movies[row, 0] = sorted_users[start:stop].copy()
        fit_movies[row, 1] = sorted_ratings[start:stop].copy()

    empty_movie_groups = np.empty((0, 2), dtype=object)
    return SplitData(
        users_train=fit_users,
        users_test=validation_users,  # Validation, NOT the original test set.
        movies_train=fit_movies,
        movies_test=empty_movie_groups,  # Not needed by RMSE evaluation.
        users_indices_train=fit_user_ids,
        users_indices_test=validation_user_ids,
        movies_indices_train=fit_movie_ids,
        movies_indices_test=np.empty(0, dtype=np.int64),
        n_users=n_users,
        n_movies=n_movies,
    )


tuning_split = build_validation_split(
    train_groups, train_user_ids, n_users, n_movies,
    fraction=VALIDATION_FRACTION, seed=SEED + 1,
)
fit_count = sum(len(row[1]) for row in tuning_split.users_train)
validation_count = sum(len(row[1]) for row in tuning_split.users_test)
print(f"Tuning fit interactions: {fit_count:,}")
print(f"Tuning validation interactions: {validation_count:,}")
print("Original test interactions used during tuning: 0")

Tuning fit interactions: 23,046,935
Tuning validation interactions: 2,554,804
Original test interactions used during tuning: 0


## 4. Bias-Only ALS hyperparameter search

In [5]:
def objective_bias(trial: optuna.Trial) -> float:
    lambda_reg = trial.suggest_float("lambda_reg", 1e-4, 1.0, log=True)
    gamma = trial.suggest_float("gamma", 1e-5, 1e-1, log=True)
    params = {"lambda_reg": lambda_reg, "gamma": gamma, "tau": runner.tau}

    # Reuse the SAME ALS loop, loss, RMSE, and initialization as main.py.
    history, _ = runner._train_bias_only(tuning_split, params)
    return float(history["rmse"]["test"][-1])  # Here 'test' means validation.


study_bias = optuna.create_study(
    study_name=f"{DATASET_NAME}_bias_only",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_bias.optimize(objective_bias, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best Bias-Only validation RMSE: {study_bias.best_value:.6f}")
print("Best Bias-Only parameters:", study_bias.best_params)

# A plain dictionary is supported directly by main.py's _load_parameter_file().
utils.save_data_to_pickle(
    str(runner.model_path), study_bias.best_params, "params_bias", log=False
)
# Also preserve the full Optuna trials for later analysis.
utils.save_data_to_pickle(
    str(runner.model_path), study_bias, "study_bias", log=False
)
print("Saved params_bias.pkl and study_bias.pkl")

Best trial: 11. Best value: 0.856507: 100%|██████████| 20/20 [22:11<00:00, 66.57s/it]

Best Bias-Only validation RMSE: 0.856507
Best Bias-Only parameters: {'lambda_reg': 0.02347101288568332, 'gamma': 0.007631550861559506}
Saved params_bias.pkl and study_bias.pkl


## 5. Bias + Latent Factors ALS hyperparameter search

In [ ]:
def objective_bias_factors(trial: optuna.Trial) -> float:
    lambda_reg = trial.suggest_float("lambda_reg", 1e-4, 1.0, log=True)
    gamma = trial.suggest_float("gamma", 1e-5, 1e-1, log=True)
    tau = trial.suggest_float("tau", 0.5, 1.0)
    params = {"lambda_reg": lambda_reg, "gamma": gamma, "tau": tau}

    # main.py's factor method uses np.random.default_rng(SEED), K=N_FACTORS,
    # and zero factors for any entity absent from tuning training interactions.
    history, _ = runner._train_factor_model(tuning_split, params)
    return float(history["rmse"]["test"][-1])  # Here 'test' means validation.


study_bias_factors = optuna.create_study(
    study_name=f"{DATASET_NAME}_bias_factors_k{N_FACTORS}",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_bias_factors.optimize(
    objective_bias_factors, n_trials=N_TRIALS, show_progress_bar=True
)

print(f"Best Matrix Factorization validation RMSE: {study_bias_factors.best_value:.6f}")
print("Best Matrix Factorization parameters:", study_bias_factors.best_params)

utils.save_data_to_pickle(
    str(runner.model_path), study_bias_factors.best_params,
    "params_bias_factors", log=False
)
utils.save_data_to_pickle(
    str(runner.model_path), study_bias_factors,
    "study_bias_factors", log=False
)
print("Saved params_bias_factors.pkl and study_bias_factors.pkl")

## 6. Trial visualizations (PDF and PNG)

In [ ]:
def plot_trials(study: optuna.Study, model_label: str, filename: str) -> None:
    """
    Plot completed Optuna trials, highlighting the minimum validation RMSE.
    """
    completed = [
        trial for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and trial.value is not None
    ]
    if not completed:
        raise ValueError("No completed trials to visualize.")

    numbers = np.asarray([trial.number + 1 for trial in completed])
    values = np.asarray([trial.value for trial in completed], dtype=float)
    best = study.best_trial

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(numbers, values, "o-", color="tab:blue", label="Trial RMSE")
    ax.scatter([best.number + 1], [best.value], color="black", s=90,
               zorder=4, label="Best trial")
    for trial in completed:
        ax.annotate(str(trial.number + 1), (trial.number + 1, trial.value),
                    xytext=(4, 5), textcoords="offset points", fontsize=9)
    ax.set(
        title=f"Optuna: {model_label}",
        xlabel="Trial number",
        ylabel="Validation RMSE (lower is better)",
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    pdf_file = image_path / f"{filename}.pdf"
    fig.savefig(pdf_file, format="pdf", bbox_inches="tight")
    fig.savefig(pdf_file.with_suffix(".png"), format="png", dpi=200,
                bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {pdf_file.name} and {pdf_file.with_suffix('.png').name}")


plot_trials(study_bias, "Bias-Only", f"parameter_search_{DATASET_NAME}_bias")
plot_trials(
    study_bias_factors, f"Bias + Factors (K={N_FACTORS})",
    f"parameter_search_{DATASET_NAME}_k_{N_FACTORS}",
)

## 7. Check best parameters and train final models with `main.py`

In [ ]:
# Verify the saved files use lowercase names recognized by current main.py.
bias_params = utils.load_data_from_pickle(
    str(runner.model_path), "params_bias", log=False
)
factor_params = utils.load_data_from_pickle(
    str(runner.model_path), "params_bias_factors", log=False
)
assert set(bias_params) == {"lambda_reg", "gamma"}
assert set(factor_params) == {"lambda_reg", "gamma", "tau"}

print("Bias-Only parameters:", bias_params)
print("Factor-model parameters:", factor_params)
print("Validation RMSE (bias-only):", study_bias.best_value)
print("Validation RMSE (factors):", study_bias_factors.best_value)
print("Parameter files are ready for main.py.")